In [1]:
#####################
## IMPORT
#####################
import os 
import shutil
import json

from ultralytics import YOLO

In [2]:
## Récupérer les fichiers hors de val et train : 
def get_files(path):
    files = []
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            if filename.endswith('.jpg'):
                files.append(filename)
    return files

train_val_files = get_files('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/data/split')

In [3]:
non_used = os.listdir('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test YOLO/nonutilisé/images')
len(non_used)

16267

In [4]:
len([i for i in non_used if i not in train_val_files])
## les deux sont égaux, donc pas de doublon

16267

In [5]:
## Tout est ok : on récupère un sample de 100 images
sample_test = [i for i in non_used if i not in train_val_files][:400]

In [6]:
## On crée un dossier pour les images de test
os.makedirs('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images', exist_ok=True)
## On copie les images dans le dossier
for image in sample_test:
    src = os.path.join('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test YOLO/nonutilisé/images', image)
    dst = os.path.join('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images', image)
    shutil.copy(src, dst)


In [7]:
## création du fichier ground_truth_test.json à partir du json de départ
with open('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/ign25synth_train.json', 'r') as f:
    data = json.load(f)


In [12]:
test_gt =[i for i in data if i['image'].split('/')[-1] in sample_test]

In [13]:
# Modifier le nom de l'image dans chaque entrée
for img in test_gt:
    img['image'] = os.path.join('test', img['image'].split('/')[-1])

In [14]:
## export json gt
with open('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/ground_truth_test_400.json', 'w') as f:
    json.dump(test_gt, f, indent=4)

In [17]:
###YOLO

from ultralytics import YOLO

model = YOLO('/Users/rolly/Downloads/results(1)/runs/detect/yolo_ign25synth_bbox_finetuning2/weights/best.pt')
preds = model.predict(source='/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/', save=True, save_txt=True, save_conf=True)


image 1/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000146.jpg: 640x640 85 imgs, 94.7ms
image 2/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000152.jpg: 640x640 3 imgs, 71.2ms
image 3/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000185.jpg: 640x640 4 imgs, 68.4ms
image 4/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000191.jpg: 640x640 3 imgs, 57.5ms
image 5/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000218.jpg: 640x640 200 imgs, 68.9ms
image 6/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000224.jpg: 640x640 8 imgs, 61.4ms
image 7/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000230.jpg: 640x640 13 imgs, 71.5ms
image 8/400 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_image

In [18]:
## fonction onversion output YOLO dans le même format
## Format attendu :
'''
{'image': 'test/000146.jpg',
  'groups': [[{'vertices': [[268.7747073036216, 1885.3678675115198],
      [299.32158230346687, 1885.3678675115198],
      [299.32158230346687, 1916.3522425127353],
      [268.7747073036216, 1916.3522425127353],
      [268.7747073036216, 1885.3678675115198]],
     'text': '112',
     'illegible': False,
     'truncated': False}],
   [{'vertices': [[136.95474884478415, 1934.4083697779886],
      [167.9391238449445, 1934.4083697779886],
      [167.9391238449445, 1964.955244778889],
      [136.95474884478415, 1964.955244778889],
      [136.95474884478415, 1934.4083697779886]],
     'text': '111',
     'illegible': False,
     'truncated': False}],]}
'''

def conversion_bbox_vertices(bbox, image_size=2000):
    '''
    Convertit une bbox YOLO en liste des sommets du rectangle en coordonnées absolues.

    Args:
        bbox (str): bbox au format YOLO "classe x_center y_center width height"
        image_size (int): taille de l'image (carrée) pour dénormaliser (par défaut : 2000)

    Returns:
        list: liste des 5 sommets (le dernier répété pour fermer le polygone)
    '''
    coords = bbox.strip().split()[1:-1]  # On ignore la classe et la confiance
    x_center, y_center, width, height = [float(coord) * image_size for coord in coords]

    # Calcul des coins
    x_min = x_center - width / 2
    x_max = x_center + width / 2
    y_min = y_center - height / 2
    y_max = y_center + height / 2

    # Sommets dans l'ordre horaire (ou antihoraire) + fermeture du polygone
    vertices = [
        [x_min, y_min],
        [x_max, y_min],
        [x_max, y_max],
        [x_min, y_max],
        [x_min, y_min]
    ]
    return vertices


#Chaque image dans un dico, avec groups qui contient une liste de liste de dictionnaires. Ces dictionaries contiennent les arrêtes, le texte, et les flags illegible et truncated.
def conversion_yolo_output(yolo_output):
    '''
    Convertit l'output de YOLO dans le format attendu pour le fichier ground_truth_test.json.
    yolo_output : str
        Le chemin vers le fichier de sortie de YOLO
    '''
    txt_preds = [i for i in os.listdir(yolo_output) if i.endswith('.txt')]
    preds = {
        f'{txt.replace("txt","jpg")}' : open(os.path.join(yolo_output, txt), 'r').readlines() for txt in txt_preds
    }
    formated_data =[]
    for img,pred in preds.items():
        groups = []
        for bbox in pred:
            vertices = conversion_bbox_vertices(bbox)
            group = [{
                'vertices': vertices,
                'text':'',  
              #  'illegible': False,  # On peut ajuster selon les besoins
               # 'truncated': False  # On peut ajuster selon les besoins
            }]
            groups.append(group)
        formated_data.append({'image': f'test/{img}', 'groups': groups})

    return formated_data

In [19]:
yolo_output = '/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/detect/predict3/labels'
converted_output = conversion_yolo_output(yolo_output)

In [20]:
len(converted_output)

399

In [22]:
## Export du fichier json 
with open('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/pred_test_400.json', 'w') as f:
    json.dump(converted_output, f, indent=4)

In [ ]:
## installation : https://github.com/icdar-maptext/evaluation/tree/main?tab=readme-ov-file#conda-installation
#Installation en CLI uniquement
! python3 evaluation/eval.py --gt evaluation/json/ground_truth_test.json --pred evaluation/json/pred_test.json --task det 

In [ ]:
#Results sur 100 images de test :
{'recall': 0.7240005495260338, 'precision': 0.8876536971534446, 'fscore': 0.7975181598062955, 'tightness': 0.7904895282339552, 'quality': 0.6304297539032906, 'hmean': 0.7951614358718819}
##Résultats sur 400 images de test :
{'recall': 0.6815001492190867, 'precision': 0.8900437399852756, 'fscore': 0.7719350961538463, 'tightness': 0.7990308044828535, 'quality': 0.6167999208883567, 'hmean': 0.7807604789778504}